In [0]:
def drop_fivetran_metadata(table_name):
    """
    Drop Fivetran metadata columns from a table.
    
    Args:
        table_name (str): Fully qualified table name (catalog.schema.table)
    
    Returns:
        DataFrame: Cleaned DataFrame without Fivetran metadata columns
    """
    # Define Fivetran metadata columns to drop
    fivetran_columns = ['_file', '_line', '_modified', '_fivetran_synced']
    
    # Read the table
    df = spark.table(table_name)
    
    # Get existing columns
    existing_cols = df.columns
    
    # Find which Fivetran columns exist in the table
    cols_to_drop = [col for col in fivetran_columns if col in existing_cols]
    
    # Drop the metadata columns if they exist
    if cols_to_drop:
        df_cleaned = df.drop(*cols_to_drop)
        print(f"Dropped columns from {table_name}: {cols_to_drop}")
    else:
        df_cleaned = df
        print(f"No Fivetran metadata columns found in {table_name}")
    
    return df_cleaned

# Example usage:
df_customers_cleaned = drop_fivetran_metadata("bronze_catalog.bronze_raw_schema.customers")
display(df_customers_cleaned)

In [0]:
# Get all tables in the bronze schema
tables = spark.sql("SHOW TABLES IN bronze_catalog.bronze_raw_schema").collect()

# Process each table
for table_row in tables:
    table_name = table_row.tableName
    source_table = f"bronze_catalog.bronze_raw_schema.{table_name}"
    target_table = f"bronze_catalog.bronze_stage_schema.{table_name}"
    
    print(f"\nProcessing: {source_table}")
    print("=" * 60)
    
    # Clean the table
    df_cleaned = drop_fivetran_metadata(source_table)
    
    # Write to bronze_stage_schema
    print(f"Writing to: {target_table}")
    df_cleaned.write.mode("overwrite").saveAsTable(target_table)
    
    # Show row count
    row_count = df_cleaned.count()
    print(f"✓ Successfully wrote {row_count} rows to {target_table}")
    print()

print("\n" + "=" * 60)
print("All tables processed and stored in bronze_catalog.bronze_stage_schema")
print("=" * 60)